# FocusFlow Agent — Milestone 1 Notebook

**Goal:** Build the first working version of **FocusFlow**, a personal planning AI agent.

In this milestone, FocusFlow takes a messy task dump and produces:

1. A prioritized task table  
2. A realistic daily schedule  
3. A plan/risk check  
4. A next best action  
5. A Slack-message preview  
6. Calendar-ready hold previews  

No Slack integration yet. No real Google Calendar integration yet. Those come in Milestones 2 and 3.


## 1. Setup

Run this cell first. It installs the small set of packages needed for Milestone 1.

- `openai`: optional, only needed when you turn off mock mode and call a real LLM
- `pandas`: displays task and schedule tables cleanly


In [ ]:
# Install dependencies.
# In Colab, this usually takes a few seconds.
!pip install openai pandas -q

## 2. Imports and configuration

This notebook is designed to be workshop-safe.

By default, `USE_MOCK_MODE = True`, so the notebook runs without any API key. This makes it easy to test the full flow in a live room.

When you are ready to call a real model, set:

```python
USE_MOCK_MODE = False
```

Then enter your OpenAI API key when prompted.


In [ ]:
import os
import json
import re
from datetime import datetime
from zoneinfo import ZoneInfo
from getpass import getpass

import pandas as pd
from IPython.display import display, Markdown

# Workshop-safe default.
# Keep this True while teaching the notebook flow.
# Set to False when you want to call a real OpenAI model.
USE_MOCK_MODE = True

# You can change this to any model available in your OpenAI account.
MODEL_NAME = "gpt-4.1-mini"

TIMEZONE = "America/Los_Angeles"
TODAY_DATE = datetime.now(ZoneInfo(TIMEZONE)).date().isoformat()

print(f"FocusFlow notebook loaded. Today: {TODAY_DATE}. Timezone: {TIMEZONE}.")
print(f"Mock mode: {USE_MOCK_MODE}")

FocusFlow notebook loaded. Today: 2026-05-16. Timezone: America/Los_Angeles.
Mock mode: True


## 3. Optional: API key setup

Only run this cell if you set `USE_MOCK_MODE = False`.

For the first workshop run, you can leave mock mode on and skip this cell.


In [ ]:
if not USE_MOCK_MODE:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
    print("API key configured.")
else:
    print("Mock mode is ON. No API key needed for Milestone 1 demo.")

Mock mode is ON. No API key needed for Milestone 1 demo.


## 4. Sample messy task dump

This is the input FocusFlow will organize.

For the workshop, start with this sample so everyone sees the same output first. Later, attendees can replace it with their own task dump.


In [ ]:
task_dump = """
I need to prepare slides for Friday, email Sam, review an AI paper,
book a dentist appointment, finish project update, go to the gym,
pay rent, and plan the meetup agenda.

I have 4 hours today and prefer deep work in the morning.
""".strip()

print(task_dump)

I need to prepare slides for Friday, email Sam, review an AI paper,
book a dentist appointment, finish project update, go to the gym,
pay rent, and plan the meetup agenda.

I have 4 hours today and prefer deep work in the morning.


## 5. FocusFlow agent workflow

FocusFlow is more than a prompt that rewrites a to-do list.

It follows a simple agentic planning loop:

```text
Extract → Categorize → Prioritize → Schedule → Validate → Preview Integrations
```

This is the core teaching point of Milestone 1.


## 6. Define the expected output format

A high-quality agent should return predictable structured output.

This schema gives us a contract between the AI planning step and the later integrations.


In [ ]:
EXPECTED_OUTPUT_FORMAT = {
    "tasks": [
        {
            "task": "string",
            "category": "Work | Personal | Admin | Health | Learning | Community | Other",
            "priority": "High | Medium | Low",
            "effort": "High | Medium | Low",
            "urgency": "High | Medium | Low",
            "when": "Today | This Week | Later",
            "estimated_minutes": 30,
            "reason": "short explanation"
        }
    ],
    "schedule": [
        {
            "title": "string",
            "start_time": "ISO 8601 datetime",
            "end_time": "ISO 8601 datetime",
            "type": "deep_work | admin | personal | break | flexible",
            "reason": "short explanation"
        }
    ],
    "risks": ["string"],
    "next_best_action": "string",
    "slack_message": "string",
    "calendar_preview": [
        {
            "summary": "string",
            "start": "ISO 8601 datetime",
            "end": "ISO 8601 datetime"
        }
    ]
}

EXPECTED_OUTPUT_FORMAT

{'tasks': [{'task': 'string',
   'category': 'Work | Personal | Admin | Health | Learning | Community | Other',
   'priority': 'High | Medium | Low',
   'effort': 'High | Medium | Low',
   'urgency': 'High | Medium | Low',
   'when': 'Today | This Week | Later',
   'estimated_minutes': 30,
   'reason': 'short explanation'}],
 'schedule': [{'title': 'string',
   'start_time': 'ISO 8601 datetime',
   'end_time': 'ISO 8601 datetime',
   'type': 'deep_work | admin | personal | break | flexible',
   'reason': 'short explanation'}],
 'risks': ['string'],
 'next_best_action': 'string',
 'slack_message': 'string',
 'calendar_preview': [{'summary': 'string',
   'start': 'ISO 8601 datetime',
   'end': 'ISO 8601 datetime'}]}

## 7. Define the FocusFlow prompt

This is the core agent instruction.

The prompt tells FocusFlow exactly how to reason about the messy task dump and exactly what format to return.


In [ ]:
FOCUSFLOW_SYSTEM_PROMPT = """
You are FocusFlow, a personal planning AI agent.

Your goal is to turn a messy task dump into a realistic daily plan.

You must follow this workflow:
1. Extract individual tasks from the messy input.
2. Categorize each task.
3. Estimate urgency, effort, and importance.
4. Prioritize tasks into Today, This Week, and Later.
5. Create a realistic schedule for today based on the user's available time and preferences.
6. Flag risks such as overload, missing deadlines, unclear tasks, or too many deep-work items.
7. Recommend the single next best action.
8. Generate a Slack-ready daily plan message.
9. Generate calendar-ready event previews.

Planning rules:
- Do not schedule more work than the available time allows.
- Prefer deep work in the morning if the user requests it.
- Batch small admin tasks together.
- Include breaks between deep-work blocks.
- If a task has an explicit deadline, treat it as more urgent.
- If a task has no deadline, infer priority conservatively.
- If the plan is overloaded, move lower-priority tasks to This Week or Later.
- Be practical, concise, and realistic.

Return valid JSON only.
""".strip()


def build_user_prompt(task_dump, today_date, timezone):
    """Build the user prompt passed to the LLM."""
    return f"""
Task dump:
{task_dump}

Today's date:
{today_date}

Timezone:
{timezone}

Return JSON with exactly these keys:
- tasks
- schedule
- risks
- next_best_action
- slack_message
- calendar_preview

Each task must include:
- task
- category
- priority
- effort
- urgency
- when
- estimated_minutes
- reason

Each schedule item must include:
- title
- start_time
- end_time
- type
- reason

Each calendar_preview item must include:
- summary
- start
- end
""".strip()

print(FOCUSFLOW_SYSTEM_PROMPT[:500] + "...")

You are FocusFlow, a personal planning AI agent.

Your goal is to turn a messy task dump into a realistic daily plan.

You must follow this workflow:
1. Extract individual tasks from the messy input.
2. Categorize each task.
3. Estimate urgency, effort, and importance.
4. Prioritize tasks into Today, This Week, and Later.
5. Create a realistic schedule for today based on the user's available time and preferences.
6. Flag risks such as overload, missing deadlines, unclear tasks, or too many deep-...


## 8. Helper functions

These helpers make the notebook robust:

- `extract_json`: handles model output that accidentally includes markdown fences
- `validate_plan`: checks that the plan has the required structure
- `make_mock_plan`: gives us a reliable demo output without calling an API


In [ ]:
def extract_json(raw_text):
    """Extract JSON from a model response.

    Some models return JSON wrapped in ```json fences. This function strips
    those fences and parses the JSON safely.
    """
    text = raw_text.strip()

    # Remove common markdown code fences if present.
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return json.loads(text)


def validate_plan(plan, verbose=True):
    """Validate the core FocusFlow plan structure before display or integration."""
    required_keys = [
        "tasks",
        "schedule",
        "risks",
        "next_best_action",
        "slack_message",
        "calendar_preview",
    ]

    missing = [key for key in required_keys if key not in plan]
    if missing:
        if verbose:
            print("Missing required keys:", missing)
        return False

    if not isinstance(plan["tasks"], list):
        if verbose:
            print("Expected 'tasks' to be a list.")
        return False

    if not isinstance(plan["schedule"], list):
        if verbose:
            print("Expected 'schedule' to be a list.")
        return False

    if not isinstance(plan["risks"], list):
        if verbose:
            print("Expected 'risks' to be a list.")
        return False

    if not isinstance(plan["calendar_preview"], list):
        if verbose:
            print("Expected 'calendar_preview' to be a list.")
        return False

    if verbose:
        print("Plan structure looks good.")
    return True


def iso_at(today_date, hour, minute=0, timezone=TIMEZONE):
    """Create an ISO 8601 datetime string for a given date/time/timezone."""
    tz = ZoneInfo(timezone)
    y, m, d = map(int, today_date.split("-"))
    dt = datetime(y, m, d, hour, minute, tzinfo=tz)
    return dt.isoformat()


def make_mock_plan(today_date=TODAY_DATE, timezone=TIMEZONE):
    """Return a deterministic example plan for demo and testing.

    This lets the workshop run smoothly even without API keys.
    """
    return {
        "tasks": [
            {
                "task": "Finish project update",
                "category": "Work",
                "priority": "High",
                "effort": "High",
                "urgency": "High",
                "when": "Today",
                "estimated_minutes": 90,
                "reason": "Important work task and likely needed before other updates."
            },
            {
                "task": "Prepare slides for Friday",
                "category": "Work",
                "priority": "High",
                "effort": "High",
                "urgency": "High",
                "when": "Today",
                "estimated_minutes": 75,
                "reason": "Explicit deadline makes this time-sensitive."
            },
            {
                "task": "Plan the meetup agenda",
                "category": "Community",
                "priority": "High",
                "effort": "Medium",
                "urgency": "Medium",
                "when": "Today",
                "estimated_minutes": 45,
                "reason": "Useful to make progress while planning context is fresh."
            },
            {
                "task": "Pay rent",
                "category": "Admin",
                "priority": "High",
                "effort": "Low",
                "urgency": "High",
                "when": "Today",
                "estimated_minutes": 10,
                "reason": "Quick task with potentially high consequence if delayed."
            },
            {
                "task": "Email Sam",
                "category": "Admin",
                "priority": "Medium",
                "effort": "Low",
                "urgency": "Medium",
                "when": "Today",
                "estimated_minutes": 15,
                "reason": "Quick communication task that can be batched with admin work."
            },
            {
                "task": "Go to the gym",
                "category": "Health",
                "priority": "Medium",
                "effort": "Medium",
                "urgency": "Medium",
                "when": "Today",
                "estimated_minutes": 60,
                "reason": "Health task fits better outside the 4-hour work planning window."
            },
            {
                "task": "Review an AI paper",
                "category": "Learning",
                "priority": "Medium",
                "effort": "High",
                "urgency": "Low",
                "when": "This Week",
                "estimated_minutes": 60,
                "reason": "Valuable but not as urgent as deadline-driven work."
            },
            {
                "task": "Book dentist appointment",
                "category": "Personal",
                "priority": "Low",
                "effort": "Low",
                "urgency": "Low",
                "when": "This Week",
                "estimated_minutes": 10,
                "reason": "Quick personal admin task that can be done later this week."
            },
        ],
        "schedule": [
            {
                "title": "Finish project update",
                "start_time": iso_at(today_date, 9, 0, timezone),
                "end_time": iso_at(today_date, 10, 30, timezone),
                "type": "deep_work",
                "reason": "Use morning focus for the highest-priority deep-work task."
            },
            {
                "title": "Break",
                "start_time": iso_at(today_date, 10, 30, timezone),
                "end_time": iso_at(today_date, 10, 45, timezone),
                "type": "break",
                "reason": "Short reset between deep-work blocks."
            },
            {
                "title": "Prepare slide outline",
                "start_time": iso_at(today_date, 10, 45, timezone),
                "end_time": iso_at(today_date, 12, 0, timezone),
                "type": "deep_work",
                "reason": "Deadline-driven work benefits from protected focus time."
            },
            {
                "title": "Admin batch: pay rent + email Sam",
                "start_time": iso_at(today_date, 14, 0, timezone),
                "end_time": iso_at(today_date, 14, 30, timezone),
                "type": "admin",
                "reason": "Batch low-effort admin tasks together."
            },
            {
                "title": "Plan meetup agenda",
                "start_time": iso_at(today_date, 14, 30, timezone),
                "end_time": iso_at(today_date, 15, 15, timezone),
                "type": "flexible",
                "reason": "Medium-effort planning task fits after urgent work is complete."
            },
        ],
        "risks": [
            "The full task list exceeds the 4-hour planning window, so lower-priority work should move to later this week.",
            "Reviewing the AI paper is a deep-work task and should not be squeezed into an already full day.",
            "Prepare slides needs a clear definition of done, such as outline only vs. full deck."
        ],
        "next_best_action": "Start with the project update before opening email or doing smaller admin tasks.",
        "slack_message": """*FocusFlow Daily Plan*\n\n*Top priorities*\n1. Finish project update\n2. Prepare slide outline\n3. Pay rent + email Sam\n\n*Schedule*\n• 9:00–10:30 — Finish project update\n• 10:45–12:00 — Prepare slide outline\n• 2:00–2:30 — Admin batch\n• 2:30–3:15 — Plan meetup agenda\n\n*Risk*\nYour list is larger than today's 4-hour planning window. Move paper review and dentist booking to later this week.\n\n*Next best action*\nStart with the project update.""",
        "calendar_preview": [
            {
                "summary": "FocusFlow: Finish project update",
                "start": iso_at(today_date, 9, 0, timezone),
                "end": iso_at(today_date, 10, 30, timezone)
            },
            {
                "summary": "FocusFlow: Prepare slide outline",
                "start": iso_at(today_date, 10, 45, timezone),
                "end": iso_at(today_date, 12, 0, timezone)
            },
            {
                "summary": "FocusFlow: Admin batch: pay rent + email Sam",
                "start": iso_at(today_date, 14, 0, timezone),
                "end": iso_at(today_date, 14, 30, timezone)
            },
            {
                "summary": "FocusFlow: Plan meetup agenda",
                "start": iso_at(today_date, 14, 30, timezone),
                "end": iso_at(today_date, 15, 15, timezone)
            }
        ]
    }


## 9. Generate the FocusFlow plan

This function supports two modes:

- **Mock mode:** deterministic demo output, no API key required
- **Real mode:** calls the OpenAI Responses API and parses JSON output

For a live workshop, start with mock mode. Then show how to turn real mode on.


In [ ]:
def generate_focusflow_plan(task_dump, today_date=TODAY_DATE, timezone=TIMEZONE, use_mock=USE_MOCK_MODE):
    """Generate a FocusFlow plan from a messy task dump."""
    if use_mock:
        return make_mock_plan(today_date=today_date, timezone=timezone)

    # Import OpenAI only when real mode is used, so mock mode stays lightweight.
    from openai import OpenAI

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

    response = client.responses.create(
        model=MODEL_NAME,
        input=[
            {"role": "system", "content": FOCUSFLOW_SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(task_dump, today_date, timezone)},
        ],
        temperature=0.2,
    )

    return extract_json(response.output_text)


plan = generate_focusflow_plan(task_dump)
validate_plan(plan)

# Show raw JSON preview.
print(json.dumps(plan, indent=2)[:2000] + "\n...")

Plan structure looks good.
{
  "tasks": [
    {
      "task": "Finish project update",
      "category": "Work",
      "priority": "High",
      "effort": "High",
      "urgency": "High",
      "when": "Today",
      "estimated_minutes": 90,
      "reason": "Important work task and likely needed before other updates."
    },
    {
      "task": "Prepare slides for Friday",
      "category": "Work",
      "priority": "High",
      "effort": "High",
      "urgency": "High",
      "when": "Today",
      "estimated_minutes": 75,
      "reason": "Explicit deadline makes this time-sensitive."
    },
    {
      "task": "Plan the meetup agenda",
      "category": "Community",
      "priority": "High",
      "effort": "Medium",
      "urgency": "Medium",
      "when": "Today",
      "estimated_minutes": 45,
      "reason": "Useful to make progress while planning context is fresh."
    },
    {
      "task": "Pay rent",
      "category": "Admin",
      "priority": "High",
      "effort": "Low",

## 10. Prioritized task table

This table is the first high-value output.

It shows how FocusFlow turned messy text into structured tasks with categories, priorities, urgency, effort, and reasoning.


In [ ]:
tasks_df = pd.DataFrame(plan["tasks"])

task_columns = [
    "task",
    "category",
    "priority",
    "effort",
    "urgency",
    "when",
    "estimated_minutes",
    "reason",
]

tasks_df = tasks_df[task_columns]
display(tasks_df)

,task,category,priority,effort,urgency,when,estimated_minutes,reason
0,Finish project update,Work,High,High,High,Today,90,Important work task and likely needed before o...
1,Prepare slides for Friday,Work,High,High,High,Today,75,Explicit deadline makes this time-sensitive.
2,Plan the meetup agenda,Community,High,Medium,Medium,Today,45,Useful to make progress while planning context...
3,Pay rent,Admin,High,Low,High,Today,10,Quick task with potentially high consequence i...
4,Email Sam,Admin,Medium,Low,Medium,Today,15,Quick communication task that can be batched w...
5,Go to the gym,Health,Medium,Medium,Medium,Today,60,Health task fits better outside the 4-hour wor...
6,Review an AI paper,Learning,Medium,High,Low,This Week,60,Valuable but not as urgent as deadline-driven ...
7,Book dentist appointment,Personal,Low,Low,Low,This Week,10,Quick personal admin task that can be done lat...


## 11. Today’s schedule

This is the second high-value output.

The agent does not just list tasks. It turns selected tasks into a realistic schedule.


In [ ]:
schedule_df = pd.DataFrame(plan["schedule"])

schedule_columns = [
    "title",
    "start_time",
    "end_time",
    "type",
    "reason",
]

schedule_df = schedule_df[schedule_columns]
display(schedule_df)

,title,start_time,end_time,type,reason
0,Finish project update,2026-05-16T09:00:00-07:00,2026-05-16T10:30:00-07:00,deep_work,Use morning focus for the highest-priority dee...
1,Break,2026-05-16T10:30:00-07:00,2026-05-16T10:45:00-07:00,break,Short reset between deep-work blocks.
2,Prepare slide outline,2026-05-16T10:45:00-07:00,2026-05-16T12:00:00-07:00,deep_work,Deadline-driven work benefits from protected f...
3,Admin batch: pay rent + email Sam,2026-05-16T14:00:00-07:00,2026-05-16T14:30:00-07:00,admin,Batch low-effort admin tasks together.
4,Plan meetup agenda,2026-05-16T14:30:00-07:00,2026-05-16T15:15:00-07:00,flexible,Medium-effort planning task fits after urgent ...


## 12. Plan check: risks and next best action

This is the validation step.

A useful planning agent should notice when the plan is overloaded, unclear, or unrealistic.


In [ ]:
display(Markdown("### Risks / Plan Check"))
for risk in plan["risks"]:
    display(Markdown(f"- {risk}"))

display(Markdown("### Next Best Action"))
display(Markdown(f"**{plan['next_best_action']}**"))

### Risks / Plan Check

- The full task list exceeds the 4-hour planning window, so lower-priority work should move to later this week.

- Reviewing the AI paper is a deep-work task and should not be squeezed into an already full day.

- Prepare slides needs a clear definition of done, such as outline only vs. full deck.

### Next Best Action

**Start with the project update before opening email or doing smaller admin tasks.**

## 13. Slack-message preview

Milestone 2 will send this message to Slack.

For Milestone 1, we only generate and preview it.


In [ ]:
display(Markdown("### Slack Daily Plan Preview"))
display(Markdown(plan["slack_message"]))

### Slack Daily Plan Preview

*FocusFlow Daily Plan*

*Top priorities*
1. Finish project update
2. Prepare slide outline
3. Pay rent + email Sam

*Schedule*
• 9:00–10:30 — Finish project update
• 10:45–12:00 — Prepare slide outline
• 2:00–2:30 — Admin batch
• 2:30–3:15 — Plan meetup agenda

*Risk*
Your list is larger than today's 4-hour planning window. Move paper review and dentist booking to later this week.

*Next best action*
Start with the project update.

## 14. Calendar hold preview

Milestone 3 will turn these previews into real Google Calendar holds.

For Milestone 1, we only generate calendar-ready blocks.


In [ ]:
calendar_df = pd.DataFrame(plan["calendar_preview"])
display(calendar_df)

,summary,start,end
0,FocusFlow: Finish project update,2026-05-16T09:00:00-07:00,2026-05-16T10:30:00-07:00
1,FocusFlow: Prepare slide outline,2026-05-16T10:45:00-07:00,2026-05-16T12:00:00-07:00
2,FocusFlow: Admin batch: pay rent + email Sam,2026-05-16T14:00:00-07:00,2026-05-16T14:30:00-07:00
3,FocusFlow: Plan meetup agenda,2026-05-16T14:30:00-07:00,2026-05-16T15:15:00-07:00


## 15. Calendar event payload preview

This is what Milestone 3 will send to the Google Calendar API.

For now, this cell only creates the payloads locally.


In [ ]:
calendar_event_payloads = []

for item in plan["calendar_preview"]:
    calendar_event_payloads.append({
        "summary": item["summary"],
        "description": "Created by FocusFlow Agent",
        "start": {
            "dateTime": item["start"],
            "timeZone": TIMEZONE,
        },
        "end": {
            "dateTime": item["end"],
            "timeZone": TIMEZONE,
        },
    })

print(json.dumps(calendar_event_payloads, indent=2))

[
  {
    "summary": "FocusFlow: Finish project update",
    "description": "Created by FocusFlow Agent",
    "start": {
      "dateTime": "2026-05-16T09:00:00-07:00",
      "timeZone": "America/Los_Angeles"
    },
    "end": {
      "dateTime": "2026-05-16T10:30:00-07:00",
      "timeZone": "America/Los_Angeles"
    }
  },
  {
    "summary": "FocusFlow: Prepare slide outline",
    "description": "Created by FocusFlow Agent",
    "start": {
      "dateTime": "2026-05-16T10:45:00-07:00",
      "timeZone": "America/Los_Angeles"
    },
    "end": {
      "dateTime": "2026-05-16T12:00:00-07:00",
      "timeZone": "America/Los_Angeles"
    }
  },
  {
    "summary": "FocusFlow: Admin batch: pay rent + email Sam",
    "description": "Created by FocusFlow Agent",
    "start": {
      "dateTime": "2026-05-16T14:00:00-07:00",
      "timeZone": "America/Los_Angeles"
    },
    "end": {
      "dateTime": "2026-05-16T14:30:00-07:00",
      "timeZone": "America/Los_Angeles"
    }
  },
  {
    "summa

## 16. Try your own messy task dump

Replace the text below with your own task dump.

This is the part attendees should personalize during the workshop.


In [ ]:
my_task_dump = """
I need to finish a project update, reply to two emails, prep for a meeting,
clean up my notes, go to the gym, buy groceries, and read one AI article.
I only have 3 hours today and I want to avoid doing deep work after 3 PM.
""".strip()

my_plan = generate_focusflow_plan(my_task_dump)
validate_plan(my_plan)

my_tasks_df = pd.DataFrame(my_plan["tasks"])[task_columns]
my_schedule_df = pd.DataFrame(my_plan["schedule"])[schedule_columns]

display(Markdown("### My Prioritized Tasks"))
display(my_tasks_df)

display(Markdown("### My Schedule"))
display(my_schedule_df)

display(Markdown("### My Next Best Action"))
display(Markdown(f"**{my_plan['next_best_action']}**"))

Plan structure looks good.


### My Prioritized Tasks

,task,category,priority,effort,urgency,when,estimated_minutes,reason
0,Finish project update,Work,High,High,High,Today,90,Important work task and likely needed before o...
1,Prepare slides for Friday,Work,High,High,High,Today,75,Explicit deadline makes this time-sensitive.
2,Plan the meetup agenda,Community,High,Medium,Medium,Today,45,Useful to make progress while planning context...
3,Pay rent,Admin,High,Low,High,Today,10,Quick task with potentially high consequence i...
4,Email Sam,Admin,Medium,Low,Medium,Today,15,Quick communication task that can be batched w...
5,Go to the gym,Health,Medium,Medium,Medium,Today,60,Health task fits better outside the 4-hour wor...
6,Review an AI paper,Learning,Medium,High,Low,This Week,60,Valuable but not as urgent as deadline-driven ...
7,Book dentist appointment,Personal,Low,Low,Low,This Week,10,Quick personal admin task that can be done lat...


### My Schedule

,title,start_time,end_time,type,reason
0,Finish project update,2026-05-16T09:00:00-07:00,2026-05-16T10:30:00-07:00,deep_work,Use morning focus for the highest-priority dee...
1,Break,2026-05-16T10:30:00-07:00,2026-05-16T10:45:00-07:00,break,Short reset between deep-work blocks.
2,Prepare slide outline,2026-05-16T10:45:00-07:00,2026-05-16T12:00:00-07:00,deep_work,Deadline-driven work benefits from protected f...
3,Admin batch: pay rent + email Sam,2026-05-16T14:00:00-07:00,2026-05-16T14:30:00-07:00,admin,Batch low-effort admin tasks together.
4,Plan meetup agenda,2026-05-16T14:30:00-07:00,2026-05-16T15:15:00-07:00,flexible,Medium-effort planning task fits after urgent ...


### My Next Best Action

**Start with the project update before opening email or doing smaller admin tasks.**

## 17. Optional: Export Milestone 1 outputs

This saves the tables as CSV files.

In Colab, you can download them from the file browser on the left.


In [ ]:
tasks_df.to_csv("focusflow_tasks.csv", index=False)
schedule_df.to_csv("focusflow_schedule.csv", index=False)
calendar_df.to_csv("focusflow_calendar_preview.csv", index=False)

print("Exported:")
print("- focusflow_tasks.csv")
print("- focusflow_schedule.csv")
print("- focusflow_calendar_preview.csv")

Exported:
- focusflow_tasks.csv
- focusflow_schedule.csv
- focusflow_calendar_preview.csv


## 18. Milestone 1 complete

You now have **FocusFlow v1**:

```text
Messy task dump
    ↓
Structured planning agent
    ↓
Prioritized task table
    ↓
Today’s schedule
    ↓
Risk check + next best action
    ↓
Slack preview + calendar preview
```

Next milestones:

- **Milestone 2:** Send the Slack daily plan message to a real Slack channel
- **Milestone 3:** Create real Google Calendar holds from the calendar preview payloads
